# tfmstudy — Colab runner

Runs the jobs of the TimesFM 3 zero-shot study and writes results to Google Drive.

1. **Runtime → Change runtime type → T4 GPU** (or connect without GPU for the CPU jobs).
2. **Runtime → Run all**.

With a GPU the job list is read from `MyDrive/timesfm-zeroshot-study/job.json`, without one from `job_cpu.json`:
`{"jobs": [{"name": ..., "job": ..., "extra_args": ...}, ...]}` (a single `{"job": ..., "extra_args": ...}` also works).
Each session takes the first job that is neither done nor running elsewhere (claims in `claims/`, refreshed every
2 minutes; a claim older than 15 minutes is considered abandoned), runs it, and moves on to the next, so several sessions
(e.g. two Google accounts sharing the folder) split the list. Jobs are resumable. The code comes from the newest
`code/tfmstudy-*.zip`, so this notebook never needs editing. From another Google account: share the folder with it and add a
shortcut named `timesfm-zeroshot-study` to its My Drive.

In [ ]:
ROOT = "/content/drive/MyDrive/timesfm-zeroshot-study"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess
assert os.path.isdir(ROOT), f'{ROOT} not found: add the shared folder (or a shortcut to it) to My Drive'
HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'no GPU: CPU jobs'
job_file = f"{ROOT}/job.json" if HAS_GPU else f"{ROOT}/job_cpu.json"
spec = json.load(open(job_file)) if os.path.exists(job_file) else {"job": "smoke", "extra_args": ""}
JOBS = spec["jobs"] if "jobs" in spec else [{"name": spec["job"], **spec}]
print(os.path.basename(job_file), '->', [j['name'] for j in JOBS])

In [ ]:
import glob, hashlib, os, zipfile
zips = sorted(glob.glob(f"{ROOT}/code/tfmstudy-*.zip"), key=os.path.getmtime)
assert zips, 'no code bundle found on Drive'
BUNDLE = zips[-1]
# The bundle name ends with the first 12 hex digits of its sha256: a corrupted upload stops here.
expected = os.path.basename(BUNDLE).rsplit('-', 1)[-1].removesuffix('.zip')
actual = hashlib.sha256(open(BUNDLE, 'rb').read()).hexdigest()[:12]
assert actual == expected, f'bundle hash mismatch: {actual} != {expected}'
print('using', BUNDLE, 'sha256 ok')
!rm -rf /content/src && mkdir -p /content/src
zipfile.ZipFile(BUNDLE).extractall('/content/src')

In [ ]:
!pip install -q -r /content/src/requirements-colab.txt 2>&1 | tail -3
!pip list 2>/dev/null | grep -iE '^(torch|timesfm|chronos-forecasting|tirex-2|statsforecast) '

In [ ]:
import datetime, shlex, threading, time, uuid
SESSION = uuid.uuid4().hex[:6]
CLAIMS = f"{ROOT}/claims"
os.makedirs(CLAIMS, exist_ok=True)
STALE, BEAT = 15 * 60, 120

def now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')

def claim(name):
    path = f"{CLAIMS}/{name}.json"
    if os.path.exists(path):
        c = json.load(open(path))
        if c.get('status') == 'done' or time.time() - os.path.getmtime(path) < STALE:
            return False
    json.dump({'status': 'running', 'session': SESSION, 'since': now()}, open(path, 'w'))
    time.sleep(10)  # let a concurrent session's claim surface, then check who holds it
    return json.load(open(path)).get('session') == SESSION

def run(job):
    path = f"{CLAIMS}/{job['name']}.json"
    out = f"{ROOT}/results/{job['job']}"
    os.makedirs(out, exist_ok=True)
    cmd = f"cd /content/src && python -m tfmstudy.{job['job']} --out {shlex.quote(out)} {job.get('extra_args', '')}"
    stop = threading.Event()
    def beat():
        while not stop.wait(BEAT):
            os.utime(path, None)
    threading.Thread(target=beat, daemon=True).start()
    log = f"{out}/run_{job['name']}_{SESSION}_{datetime.datetime.utcnow():%Y%m%dT%H%M%SZ}.log"
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='')
            fh.write(line)
        proc.wait()
    stop.set()
    status = 'done' if proc.returncode == 0 else 'failed'
    json.dump({'status': status, 'session': SESSION, 'finished': now(), 'returncode': proc.returncode}, open(path, 'w'))
    return status

for attempt in range(2):  # a second pass picks up jobs abandoned by other sessions in the meantime
    for job in JOBS:
        if claim(job['name']):
            print(f"=== {job['name']} (session {SESSION}) ===")
            print(job['name'], run(job))
print('no job left for this session')